# ACRE++ — Q-learning + optional LLM/TRL (Colab re-run)

1. **Set your clone URL** in the next cell (GitHub, HF Git, or upload this repo as a zip and unzip in Colab).
2. Run **Q-learning** to regenerate `artifacts/reward_curve.png` and `latency_curve.png` (real run).
3. Training curves + **last cell: Random vs Trained bar chart** (from `training_clean_summary.json`) — that figure is the clearest “did it learn?” snapshot.
4. **LLM + PPO** is optional (before the final chart): needs GPU + disk; if it fails, skip; Q-learning evidence is still valid.


In [ ]:
%%capture
%pip install -q matplotlib pandas torch streamlit fastapi uvicorn
%pip install -q 'transformers' 'trl' accelerate datasets sentencepiece safetensors protobuf

In [ ]:
# TODO: set your public Git URL, or use File > Upload to upload the repo, then %cd into it.
# !git clone https://github.com/YOUR_ORG/dc_scaler.git
# %cd dc_scaler
import os
assert os.path.isfile('train_clean.py'), 'Run the clone/upload step so train_clean.py is in the CWD.'

In [ ]:
import subprocess, sys, json
subprocess.check_call([sys.executable, 'train_clean.py', '--episodes', '80', '--task-id', 'incident_recovery', '--seed', '42'])
print(open('artifacts/training_clean_summary.json', encoding='utf-8').read()[:2000])

In [ ]:
from IPython.display import Image, display
from pathlib import Path
for p in [Path('artifacts/reward_curve.png'), Path('artifacts/latency_curve.png')]:
    if p.is_file():
        display(Image(str(p)))
    else:
        print("Missing:", p)

## Optional: LLM + PPO (TRL) — small model, GPU + ~1GB cache recommended
If this errors (OOM / disk), **skip**; Q-learning cells above are enough for a training pipeline demo.

In [ ]:
import os, subprocess, sys
if not os.environ.get('COLAB_GPU'):
  print('No GPU: skipping LLM training.')
else:
  subprocess.check_call([sys.executable, 'train_llm.py', '--episodes', '2', '--task-id', 'incident_recovery',
    '--model-name', 'Qwen/Qwen2.5-0.5B-Instruct', '--seed', '42', '--output-dir', 'artifacts/llm_ppo'])
  subprocess.check_call([sys.executable, 'plot_submission_artifacts.py', '--llm-log', 'artifacts/llm_ppo/llm_training_log.jsonl', '--out-dir', 'artifacts/llm_ppo'])

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

summary_path = Path("artifacts/training_clean_summary.json")
with open(summary_path, encoding="utf-8") as f:
    s = json.load(f)
r0, r1 = s["random_eval"], s["trained_eval"]

titles = [
    "Episode reward (↑ better)",
    "Total cost (↓ better)",
    "Uptime % (↑ better)",
    "Avg latency (ms / step) (↓ better)",
]
random_vals = [r0["reward"], r0["cost"], r0["uptime"], r0["avg_latency_ms"]]
trained_vals = [r1["reward"], r1["cost"], r1["uptime"], r1["avg_latency_ms"]]

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, title, vr, vt in zip(axes.ravel(), titles, random_vals, trained_vals):
    ax.bar(["Random", "Trained"], [vr, vt], color=["#c44e52", "#2c7bb6"], width=0.55)
    ax.set_title(title, fontsize=11)
    ax.grid(axis="y", alpha=0.3, linestyle="--")
fig.suptitle(
    "Same eval protocol — random baseline vs trained Q-policy",
    fontsize=13,
    fontweight="bold",
    y=1.02,
)
fig.tight_layout()
out_png = Path("artifacts/random_vs_trained_eval.png")
out_png.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_png, dpi=140, bbox_inches="tight")
plt.show()
print("Saved:", out_png.resolve())